<a href="https://colab.research.google.com/github/hirajya/CCADMACL_EXERCISES_COM222-/blob/main/Exercise2Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Exercise 2: Use Gradient Boost for Regression

Instructions:

- Use the Dataset File to train your model
- Use the Test File to generate your results
- Use the Sample Submission file to generate the same format
Submit your results to:
https://www.kaggle.com/competitions/playground-series-s4e12/overview



In [ ]:
!pip install xgboost --quiet

In [ ]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_log_error
from sklearn.compose import make_column_transformer

## Dataset
Train, test and sample submission file can be found in this link
https://www.kaggle.com/competitions/playground-series-s4e12/data

## 1. Load the Data

In [ ]:
dtr = pd.read_csv('train.csv')
dte = pd.read_csv('test.csv')
sf = pd.read_csv('sample_submission.csv')

In [ ]:
print(f"Train shape: {dtr.shape}, Test shape: {dte.shape}")

## 2. Perform Data preprocessing

In [ ]:
def extract_date_features(df, date_column):
    df[date_column] = pd.to_datetime(df[date_column])
    df[f"{date_column}_Year"] = df[date_column].dt.year
    df[f"{date_column}_Month"] = df[date_column].dt.month
    df[f"{date_column}_Day"] = df[date_column].dt.day
    df[f"{date_column}_Hour"] = df[date_column].dt.hour
    df[f"{date_column}_Minute"] = df[date_column].dt.minute
    df[f"{date_column}_Second"] = df[date_column].dt.second
    df.drop(date_column, axis=1, inplace=True)
    return df

In [ ]:
dtr = extract_date_features(dtr, 'Policy Start Date')
dte = extract_date_features(dte, 'Policy Start Date')

In [ ]:
dtr.drop(columns=['id'], inplace=True)
dte_ids = dte['id']
dte.drop(columns=['id'], inplace=True)

In [ ]:
dtr.dropna(inplace=True)

In [ ]:
X = dtr.drop(columns=['Premium Amount'])
y = dtr['Premium Amount']

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

## 3. Create a Pipeline

In [ ]:
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [ ]:
preprocessor = make_column_transformer(
    (Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ]), numerical_features),
    (Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_features)
)

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(dte)

## 4. Train the Model

In [ ]:
model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=7,
    random_state=42
)

In [ ]:
model.fit(X_train_processed, y_train)

## 5. Evaluate the Model

In [ ]:
y_val_pred = model.predict(X_val_processed)
validation_rmsle = np.sqrt(mean_squared_log_error(y_val, y_val_pred))
print(f"Validation RMSLE: {validation_rmsle}")

In [ ]:
y_test_pred = model.predict(X_test_processed)

## Generate Submission File

Choose the model that has the best performance to generate a submission file.

In [ ]:
# Create a submission DataFrame
submission_df = pd.DataFrame({
    'id': dte_ids,
    'Premium Amount': y_test_pred
})

# Save the submission file
submission_df.to_csv('submission_file.csv', index=False)
print("Submission file created: submission_file.csv")